# 04 — Matchups & Box Scores

Explore H2H category matchups: scoreboard, box scores, per-player breakdowns,
and the category win/loss/tie structure.

In [ ]:
import sys
sys.path.insert(0, "../src")
from connection import get_league
import pandas as pd

league = get_league()
print(f"Connected: {league.settings.name}")
print(f"Scoring type: {league.settings.scoring_type}")
print(f"BoxScore class: {league.BoxScoreClass.__name__}")

## Current Scoreboard

In [ ]:
scoreboard = league.scoreboard()
for m in scoreboard:
    home = m.home_team.team_name if hasattr(m.home_team, 'team_name') else str(m.home_team)
    away = m.away_team.team_name if hasattr(m.away_team, 'team_name') else str(m.away_team)
    print(f"{home:30s} vs {away:30s}  Score: {m.home_final_score}-{m.away_final_score}  Winner: {m.winner}")

# Check for category-specific data
m0 = scoreboard[0]
if hasattr(m0, 'home_team_cats') and m0.home_team_cats:
    print(f"\nCategory breakdown for first matchup:")
    print(f"  home_team_cats: {m0.home_team_cats}")
    print(f"  away_team_cats: {m0.away_team_cats}")

## Box Scores — H2H Category Structure

For H2H Category leagues, each box score has:
- `home_stats` / `away_stats`: dict of `{category: {value, result}}`
- `home_wins/losses/ties` and `away_wins/losses/ties`
- `home_lineup` / `away_lineup`: list of BoxPlayer objects

In [ ]:
box_scores = league.box_scores(matchup_period=1)
bs = box_scores[0]

print(f"Box score type: {type(bs).__name__}")
print(f"\nAll attributes:")
for attr in sorted(dir(bs)):
    if not attr.startswith('_'):
        val = getattr(bs, attr)
        if not callable(val):
            display = str(val)[:100]
            print(f"  {attr}: {display}")

In [ ]:
# Category breakdown (H2H Category leagues)
if hasattr(bs, 'home_stats') and bs.home_stats:
    home = bs.home_team.team_name if hasattr(bs.home_team, 'team_name') else 'Home'
    away = bs.away_team.team_name if hasattr(bs.away_team, 'team_name') else 'Away'

    rows = []
    for cat in bs.home_stats:
        rows.append({
            "Category": cat,
            f"{home}": bs.home_stats[cat].get('value', ''),
            "Home Result": bs.home_stats[cat].get('result', ''),
            f"{away}": bs.away_stats[cat].get('value', ''),
            "Away Result": bs.away_stats[cat].get('result', ''),
        })

    df = pd.DataFrame(rows)
    print(f"\n{home} vs {away} — Week 1 Category Breakdown:")
    display(df)

    if hasattr(bs, 'home_wins'):
        print(f"\nHome: {bs.home_wins}W-{bs.home_losses}L-{bs.home_ties}T")
        print(f"Away: {bs.away_wins}W-{bs.away_losses}L-{bs.away_ties}T")
else:
    print("No category stats — this is likely a points league.")
    if hasattr(bs, 'home_score'):
        print(f"Home score: {bs.home_score}, Away score: {bs.away_score}")

## Turnovers Inversion

In H2H Categories, **lower turnovers is better**. The team with *fewer* TOs wins that category.
Check the result field to confirm.

In [ ]:
if hasattr(bs, 'home_stats') and 'TO' in bs.home_stats:
    h_to = bs.home_stats['TO']
    a_to = bs.away_stats['TO']
    print(f"Home TO: {h_to['value']} (result: {h_to['result']})")
    print(f"Away TO: {a_to['value']} (result: {a_to['result']})")
    print(f"\nNote: Lower TO value should result in 'W'. "
          f"{'Confirmed!' if (h_to['value'] < a_to['value'] and h_to['result'] == 'W') or (h_to['value'] > a_to['value'] and a_to['result'] == 'W') or h_to['value'] == a_to['value'] else 'Check your league settings.'}")
else:
    print("TO category not found in box score stats.")

## Box Score — Player Lineups

In [ ]:
lineup = bs.home_lineup
print(f"Home lineup ({len(lineup)} players):")

rows = []
for bp in lineup:
    rows.append({
        "Name": bp.name,
        "Slot": bp.slot_position,
        "Opponent": bp.pro_opponent,
        "Game Played %": bp.game_played,
        "Points": bp.points,
    })

pd.DataFrame(rows)

In [ ]:
# Points breakdown for a box player
bp = lineup[0]
print(f"Points breakdown for {bp.name}:")
if bp.points_breakdown:
    for stat, val in sorted(bp.points_breakdown.items()):
        print(f"  {stat}: {val}")
else:
    print("  (no breakdown available)")

## Compare Box Scores Across a Full Week

In [ ]:
# Show all matchups for a given week
week = 1
all_bs = league.box_scores(matchup_period=week)
print(f"Week {week} — {len(all_bs)} matchups:\n")

for bs in all_bs:
    home = bs.home_team.team_name if hasattr(bs.home_team, 'team_name') else 'Home'
    away = bs.away_team.team_name if hasattr(bs.away_team, 'team_name') else 'Away'
    if hasattr(bs, 'home_wins'):
        print(f"{home:30s} {bs.home_wins}-{bs.home_losses}-{bs.home_ties}  vs  {bs.away_wins}-{bs.away_losses}-{bs.away_ties} {away}")
    else:
        print(f"{home:30s} vs {away}")